In [1]:
!pip install uv


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
!git clone https://github.com/DataTalksClub/machine-learning-zoomcamp.git
%cd machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
!ls -la

fatal: destination path 'machine-learning-zoomcamp' already exists and is not an empty directory.
/workspaces/machine-learning-zoomcamp-homework/05-deployment/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
total 108
drwxrwxrwx+  4 codespace codespace  4096 Oct 25 22:33 .
drwxrwxrwx+ 11 codespace codespace  4096 Oct 25 22:32 ..
-rw-rw-rw-   1 codespace codespace     8 Oct 25 22:32 .python-version
drwxrwxrwx+  4 codespace codespace  4096 Oct 25 22:32 .venv
-rw-rw-rw-   1 codespace codespace   598 Oct 25 22:32 Dockerfile
-rw-rw-rw-   1 codespace codespace  1044 Oct 25 22:32 README.md
drwxrwxrwx+  2 codespace codespace  4096 Oct 25 22:51 __pycache__
-rw-rw-rw-   1 codespace codespace   116 Oct 25 22:32 feature_defaults.json
-rw-rw-rw-   1 codespace codespace  6077 Oct 25 22:32 homework.md
-rw-rw-rw-   1 codespace codespace   563 Oct 25 22:32 homework.yaml
-rw-rw-rw-   1 codespace codespace  1456 Oct 25 22:32 model.py
-rw-rw-rw-   1 codespace codespace   872 Oct 25 22:32 mode

In [3]:
!uv sync --locked

Resolved 19 packages in 0.85ms
Checked 18 packages in 0.15ms


In [4]:
!uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


In [5]:
!sha256sum pipeline.bin

1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin


In [6]:
# Q1
%cd /workspaces/"machine-learning-zoomcamp-homework"/"05-deployment"/
!uv --version

/workspaces/machine-learning-zoomcamp-homework/05-deployment
uv 0.13.5 (x86_64-unknown-linux-gnu)


In [7]:
%cd machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
import os

if os.path.exists("pyproject.toml"):
  print("-> pyproject.toml bulundu! Doğru yerdesiniz.")
else:
  print("-> FAIL: No pyproject.toml here. Go to the right directory.")

/workspaces/machine-learning-zoomcamp-homework/05-deployment/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
-> pyproject.toml bulundu! Doğru yerdesiniz.


In [8]:
!uv sync --locked

Resolved 19 packages in 0.83ms
Checked 18 packages in 0.15ms


In [9]:
!uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


In [10]:
# Q2
import importlib.metadata
!uv sync --locked
sklearn_version = importlib.metadata.version("scikit-learn")
print(f"Downloaded scikit-learn version: {sklearn_version}")

if sklearn_version == "1.6.1":
  print("-> The right version: 1.6.1")
else:
  print(f"->Expected: 1.6.1, existing: {sklearn_version}")

Resolved 19 packages in 0.92ms
Checked 18 packages in 0.17ms
Downloaded scikit-learn version: 1.9.1
->Expected: 1.6.1, existing: 1.9.1


In [11]:
# Q3
import os
import pickle

for root, dirs, files in os.walk("."):
  if "pipeline.bin" in files:
    os.chdir(root)
    break

print("Şu anki çalışma dizini:", os.getcwd())

with open("pipeline.bin", "rb") as f_in:
  pipeline = pickle.load(f_in)

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

conversion_probability = pipeline.predict_proba([lead])[0, 1]
print(f"Soru 3 - Dönüşüm Olasılığı: {conversion_probability:.3f}")

Şu anki çalışma dizini: /workspaces/machine-learning-zoomcamp-homework/05-deployment/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment
Soru 3 - Dönüşüm Olasılığı: 0.533


/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.9

In [12]:
# Q4
!uv run uvicorn predict:app --host 0.0.0.0 --port 9696

INFO:     Started server process [32219]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 9696): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [13]:
# Q4
import pickle
with open("pipeline.bin", "rb") as f_in:
  pipeline = pickle.load(f_in)

# İkinci lead verisi (Soru 4 için)
lead_2 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

prob = pipeline.predict_proba([lead_2])[0, 1]
print(f"Soru 4 - Conversion Probability (Olasılık): {prob:.3f}")

Soru 4 - Conversion Probability (Olasılık): 0.770


In [14]:
# Q5
with open("Dockerfile", "r", encoding="utf-8") as f:
  for line in f:
    if line.startswith("FROM"):
      print(f"Base-Image: {line.strip()}")
      break

Base-Image: FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3


In [15]:
# Q6
import requests

health_url = "http://localhost:9696/health"
health_response = requests.get(health_url)

print("--- Health Endpoint Yanıtı ---")
if health_response.status_code == 200:
  print(health_response.json())
else:
  print(f"Hata Kodu: {health_response.status_code}, Mesaj: {health_response.text}")

print("\n--- Predict Endpoint Yanıtı ---")
predict_url = "http://localhost:9696/predict"
lead_data = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

predict_response = requests.post(predict_url, json=lead_data)

if predict_response.status_code == 200:
  result = predict_response.json()
  print("API Yanıtı:", result)
  for key, value in result.items():
    if "prob" in key.lower():
      print(f"Conversion Probability (Docker): {value:.3f}")
else:
  print(
      f"Hata Kodu: {predict_response.status_code}, Mesaj:"
      f" {predict_response.text}"
  )

--- Health Endpoint Yanıtı ---
{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}

--- Predict Endpoint Yanıtı ---
API Yanıtı: {'conversion_probability': 0.769799, 'conversion': True}
Conversion Probability (Docker): 0.770
